In [31]:
import os
import sys
import numpy as np
import pandas as pd
import sqlite3
from Sub04_FTIR_Analysis_Functions import Binary_to_Array

In [32]:
# Connect to the database. 
conn = sqlite3.connect('T:\\ABML\\_Projects\\RD- 2505-RA RODEO- Phase 1\\Processed Data\\FTIR\\FTIR_Database.db')
cursor = conn.cursor()
df = pd.read_sql('SELECT * FROM FTIR', conn)


In [29]:
# Go through them one-by-one and update ICO and ISO. 
BadIDs = []
for ID in df['id'].unique():
    Columns2Fetch = ['id',
                    'Deconv_CarbonylList', 'Deconv_CarbonylList_shape', 'Deconv_CarbonylList_dtype', 
                    'Deconv_SulfoxideList', 'Deconv_SulfoxideList_shape', 'Deconv_SulfoxideList_dtype', 
                    'Deconv_AliphaticList', 'Deconv_AliphaticList_shape', 'Deconv_AliphaticList_dtype', 
                    'Deconv_GaussianList', 'Deconv_GaussianList_shape', 'Deconv_GaussianList_dtype', 
                    'Deconv_ICO', 'Deconv_ISO']
    cursor.execute(f"SELECT {', '.join(Columns2Fetch)} FROM FTIR WHERE id = ?", (int(ID),))
    row = cursor.fetchall()[0]
    Carbonyl_Gaussians  = Binary_to_Array(row[1], row[2], row[3])
    Sulfoxide_Gaussians = Binary_to_Array(row[4], row[5], row[6])
    Aliphatic_Gaussians = Binary_to_Array(row[7], row[8], row[9])
    GaussianList        = Binary_to_Array(row[10], row[11], row[12])
    ICO = row[13]
    ISO = row[14]
    # Calculating the Areas. 
    CArea = (Carbonyl_Gaussians[:, 2] * np.sqrt(2 * np.pi) * np.abs(Carbonyl_Gaussians[:, 1])).sum()
    SArea = (Sulfoxide_Gaussians[:, 2] * np.sqrt(2 * np.pi) * np.abs(Sulfoxide_Gaussians[:, 1])).sum()
    AArea = (Aliphatic_Gaussians[:, 2] * np.sqrt(2 * np.pi) * np.abs(Aliphatic_Gaussians[:, 1])).sum()
    ICO2 = CArea / AArea
    ISO2 = SArea / AArea
    # Check for difference.
    if ICO == -1:
        continue
    if np.abs(ICO - ICO2) > 0.001:
        BadIDs.append(ID)
        print(f'ID={ID}, ICO Issue: ICO:{ICO:.4f}--{ICO2:.4f}; and ISO: {ISO:.4f}--{ISO2:.4f}')

C:\Users\Farhad.Abdollahi.ctr\AppData\Local\Temp\ipykernel_34768\475723411.py:22: RuntimeWarning: invalid value encountered in scalar divide
  ICO2 = CArea / AArea
C:\Users\Farhad.Abdollahi.ctr\AppData\Local\Temp\ipykernel_34768\475723411.py:23: RuntimeWarning: invalid value encountered in scalar divide
  ISO2 = SArea / AArea


In [30]:
# Fix the issue. 
for ID in BadIDs:
    Columns2Fetch = ['id',
                    'Deconv_CarbonylList', 'Deconv_CarbonylList_shape', 'Deconv_CarbonylList_dtype', 
                    'Deconv_SulfoxideList', 'Deconv_SulfoxideList_shape', 'Deconv_SulfoxideList_dtype', 
                    'Deconv_AliphaticList', 'Deconv_AliphaticList_shape', 'Deconv_AliphaticList_dtype', 
                    'Deconv_GaussianList', 'Deconv_GaussianList_shape', 'Deconv_GaussianList_dtype', 
                    'Deconv_ICO', 'Deconv_ISO']
    cursor.execute(f"SELECT {', '.join(Columns2Fetch)} FROM FTIR WHERE id = ?", (int(ID),))
    row = cursor.fetchall()[0]
    Carbonyl_Gaussians  = Binary_to_Array(row[1], row[2], row[3])
    Sulfoxide_Gaussians = Binary_to_Array(row[4], row[5], row[6])
    Aliphatic_Gaussians = Binary_to_Array(row[7], row[8], row[9])
    GaussianList        = Binary_to_Array(row[10], row[11], row[12])
    # Calculating the Areas. 
    CArea = (Carbonyl_Gaussians[:, 2] * np.sqrt(2 * np.pi) * np.abs(Carbonyl_Gaussians[:, 1])).sum()
    SArea = (Sulfoxide_Gaussians[:, 2] * np.sqrt(2 * np.pi) * np.abs(Sulfoxide_Gaussians[:, 1])).sum()
    AArea = (Aliphatic_Gaussians[:, 2] * np.sqrt(2 * np.pi) * np.abs(Aliphatic_Gaussians[:, 1])).sum()
    ICO2 = CArea / AArea
    ISO2 = SArea / AArea
    # Update the values in the Database. 
    cursor.execute("UPDATE FTIR SET Deconv_ICO = ?, Deconv_ISO = ? WHERE id = ?", (ICO2, ISO2, int(ID)))
conn.commit()
conn.close()

In [21]:
int(df['id'].unique()[0])

1

In [37]:
df[df['id'] == 243]['Deconv_CarbonylList']

242    b'a\xba|&\xcf\x9a\x9a@\x0f\x85\xe3\xfdG\xe48@\...
Name: Deconv_CarbonylList, dtype: object

In [40]:
Columns2Fetch = ['id',
                'Deconv_CarbonylList', 'Deconv_CarbonylList_shape', 'Deconv_CarbonylList_dtype', 
                'Deconv_SulfoxideList', 'Deconv_SulfoxideList_shape', 'Deconv_SulfoxideList_dtype', 
                'Deconv_AliphaticList', 'Deconv_AliphaticList_shape', 'Deconv_AliphaticList_dtype', 
                'Deconv_GaussianList', 'Deconv_GaussianList_shape', 'Deconv_GaussianList_dtype', 
                'Deconv_ICO', 'Deconv_ISO']
cursor.execute(f"SELECT {', '.join(Columns2Fetch)} FROM FTIR WHERE id = ?", (int(254),))
row = cursor.fetchall()[0]
Carbonyl_Gaussians  = Binary_to_Array(row[1], row[2], row[3])
Sulfoxide_Gaussians = Binary_to_Array(row[4], row[5], row[6])
Aliphatic_Gaussians = Binary_to_Array(row[7], row[8], row[9])
GaussianList        = Binary_to_Array(row[10], row[11], row[12])
GaussianList

array([], shape=(0, 3), dtype=float64)

In [41]:
Carbonyl_Gaussians

array([[1.69339553e+03, 9.26105456e+00, 2.08808668e-03],
       [1.71965350e+03, 1.27680508e+01, 8.59023346e-03]])